# EcoSync: Multi-Resource Exploratory Data Analysis & Model Prototyping

This notebook demonstrates the end-to-end telemetry analytics workflow of **EcoSync**:
1. **Ingestion & Validation**: Loading continuous multi-resource telemetry (Energy, Water, Waste)
2. **Exploratory Data Analysis (EDA)**: Diurnal profile characterization and cross-resource correlations
3. **Contextual Anomaly Detection**: Empirical median envelopes + Isolation Forest
4. **Demand Forecasting**: Cyclical Fourier feature transformation + Ridge Regression

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data.validate_data import validate_multi_resource_data
from src.analytics.kpi_engine import calculate_unified_campus_kpis
from src.anomaly.anomaly_service import AnomalyService
from src.forecasting.forecast_service import ForecastService

print('All EcoSync modules successfully imported!')

## 1. Load and Validate Telemetry Data

In [ ]:
data_path = project_root / 'data' / 'raw' / 'campus_multi_resource_sample.csv'
validation_result = validate_multi_resource_data(data_path)

df = validation_result['cleaned_df']
print(f'Validation Status: {validation_result["is_valid"]}')
print(f'Total Ingested Records: {len(df):,}')
print(f'Monitored Facilities: {list(df["building"].unique())}')
df.head()

## 2. Compute Unified Campus Resource KPIs

In [ ]:
kpis = calculate_unified_campus_kpis(df)
print(f"Total Energy: {kpis['energy']['total_mwh']:,.2f} MWh")
print(f"Total Water: {kpis['water']['total_m3']:,.1f} m³ (MNF: {kpis['water']['min_night_flow_m3_h']:.2f} m³/h)")
print(f"Total Waste: {kpis['waste']['total_waste_kg']:,.1f} kg (Diversion: {kpis['waste']['diversion_rate_pct']:.1f}%)")

## 3. Anomaly Detection (Contextual Baselines + Isolation Forest)

In [ ]:
anomaly_service = AnomalyService()
anomalies_df = anomaly_service.detect_multi_resource_anomalies(df)

print(f'Total Detected Incidents: {len(anomalies_df):,}')
if not anomalies_df.empty:
    severity_counts = anomalies_df['severity_tier'].value_counts()
    print('\nSeverity Breakdown:')
    print(severity_counts)
    display_cols = ['timestamp', 'building', 'resource', 'actual_value', 'expected_value', 'deviation_pct', 'severity_score']
    anomalies_df[display_cols].head(10)

## 4. Cyclical Demand Forecasting

In [ ]:
forecast_service = ForecastService()
forecast_result = forecast_service.forecast(df, building='Building_A', horizon_hours=48)

forecast_df = forecast_result.get('forecast_df', pd.DataFrame())
print(f'Forecast Generated for {len(forecast_df)} hours.')
forecast_df.head(10)